In [ ]:
!pip install tavily-python!pip install subprocess.runimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport osimport shutilimport richimport requests

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer,TextStreamermodel_name = "Qwen/Qwen3-4B"tokenizer = AutoTokenizer.from_pretrained(model_name)model = AutoModelForCausalLM.from_pretrained(    model_name,    torch_dtype="auto",    device_map="auto")class NewLineStreamer(TextStreamer):    def __init__(self, tokenizer, every=50):        super().__init__(tokenizer, skip_prompt=True,skip_special_tokens=True)        self.every = every        self.count = 0    def put(self, value):        super().put(value)        # value is a tensor/list of newly generated token ids        self.count += len(value)        if self.count >= self.every:            print("\n", end="", flush=True)            self.count = 0streamer = NewLineStreamer(tokenizer, every=35)

In [ ]:
!pip install tavily-pythonimport osfrom tavily import TavilyClient# Set TAVILY_API_KEY in your environment or Colab Secrets.tavily_api_key = os.getenv("TAVILY_API_KEY")client = TavilyClient(api_key=tavily_api_key) if tavily_api_key else None

In [ ]:
import subprocessfrom datetime import dateimport osdef edit_file(filename: str, find_str: str, replace_str: str) -> bool:    # If the file doesn't exist and find_str is empty, create the file    if not os.path.exists(filename) and find_str == "":        with open(filename, "w") as f:            f.write(replace_str)        return True    # If the file exists, read it and replace the string    try:        with open(filename, "r") as f:            content = f.read()        # Replace the string        if find_str in content:            new_content = content.replace(find_str, replace_str)            # Write the modified content back to the file            with open(filename, "w") as f:                f.write(new_content)            return True    except FileNotFoundError:        print(f"File {filename} not found and find_str is not empty.")    return Falsedef run_command(command, working_dir) -> tuple[str, int]:    try:        # Set up the process with the working directory if provided        process = subprocess.Popen(            command,            shell=True,            stdout=subprocess.PIPE,            stderr=subprocess.STDOUT,            text=True,            cwd=working_dir,        )        # Get the output and return code        output, _ = process.communicate()        error_code = process.returncode        # Clip output to first 1000 and last 1000 characters if it's longer than 2000 characters        if len(output) > 2000:            output = output[:1000] + "\n\n[...content clipped...]\n\n" + output[-1000:]        return output, error_code    except Exception as e:        return str(e), 1def list_directory(path=".") -> str:    try:        # Get the list of files and directories        items = os.listdir(path)        # Format the output        if not items:            return f"Directory '{path}' is empty."        result = f"Contents of directory '{path}':\n"        for item in items:            full_path = os.path.join(path, item)            if os.path.isdir(full_path):                item_type = "Directory"            else:                item_type = "File"            result += f"- {item} ({item_type})\n"        return result.strip()    except FileNotFoundError:        return f"Error: Directory '{path}' not found."    except PermissionError:        return f"Error: Permission denied to access '{path}'."    except Exception as e:        return f"Error listing directory '{path}': {str(e)}"def read_file_content(path: str) -> str:    try:        with open(path, "r", encoding="utf-8") as file:            content = file.read()            if len(content) > 2000:                # Get first 1000 and last 1000 characters                first_part = content[:1000]                last_part = content[-1000:]                content = first_part + "\n\n[...content clipped...]\n\n" + last_part        return content    except FileNotFoundError:        return f"Error: File '{path}' not found."    except PermissionError:        return f"Error: Permission denied to access '{path}'."    except UnicodeDecodeError:        return f"Error: Unable to decode '{path}'. The file might be binary or use an unsupported encoding."    except Exception as e:        return f"Error reading file '{path}': {str(e)}"def search_internet(query: str) -> str:    """    Retrieve the most relevant text context for the user's query    from the knowledge base.    Args:        user_prompt: The user's search query.    """    results = client.search(query=query, max_results=1)    retrieved_context = results["results"][0]["content"]    return retrieved_contextdef get_today_date() -> str:    """    Return today's date in ISO format (YYYY-MM-DD).    Returns:        str: Today's date.    """    return date.today().isoformat()

In [ ]:
tools = [    {        "type": "function",        "function": {            "name": "edit_file",            "description": "Apply a diff to a file by replacing occurrences of find_str with replace_str.",            "parameters": {                "type": "object",                "properties": {                    "filename": {                        "type": "string",                        "description": "The name of the file to modify",                    },                    "find_str": {                        "type": "string",                        "description": "The string to find in the file",                    },                    "replace_str": {                        "type": "string",                        "description": "The string to replace with",                    },                },                "required": ["filename", "find_str", "replace_str"],            },        },    },    {        "type": "function",        "function": {            "name": "run_command",            "description": "Run a shell command and return its output and error code.",            "parameters": {                "type": "object",                "properties": {                    "command": {                        "type": "string",                        "description": "The command to run in the shell",                    },                    "working_dir": {                        "type": "string",                        "description": "The working directory to run the command in",                    },                },                "required": ["command", "working_dir"],            },        },    },    {        "type": "function",        "function": {            "name": "list_directory",            "description": "List the contents of a directory.",            "parameters": {                "type": "object",                "properties": {                    "path": {                        "type": "string",                        "description": "The path to the directory to list. Defaults to the current directory.",                    },                },                "required": ["path"],            },        },    },    {        "type": "function",        "function": {            "name": "read_file_content",            "description": "Read and return the content of a file.",            "parameters": {                "type": "object",                "properties": {                    "path": {                        "type": "string",                        "description": "The path to the file to read. Defaults to the current directory.",                    },                },                "required": ["path"],            },        },    },        {        "type": "function",        "function": {            "name": "search_internet",            "description": (                    "Search the internet for up-to-date information and return the most "                    "relevant results. Use this tool whenever the user's question requires "                    "current events, recent news, live information, facts that may have "                    "changed over time, or information that is not available in the conversation."            ),            "parameters": {                "type": "object",                "properties": {                    "query": {                        "type": "string",                        "description": (                            "The search query describing what information to look for on the internet."                        ),                    }                },                "required": ["query"],            },        },    },    {    "type": "function",    "function": {        "name": "get_today_date",        "description": (            "Get today's date in ISO format (YYYY-MM-DD). "            "Use this when the user asks about the current date, today, or time-related queries."        ),        "parameters": {            "type": "object",            "properties": {},            "required": []        },    },}   ]

In [ ]:
# exit if last message doesn't have any tool calls (agent doesn't want to take any more actions)def is_goal_achieved(state) -> bool:    if len(state["messages"]) <= 2:        return False    last_message = state["messages"][-1]    return (        "role" in last_message        and last_message["role"] == "assistant"        and "tool_calls" not in last_message    )

In [ ]:
def ask_user_approval(message: str) -> bool:    user_approval = input(f"{message} (y/n): ")    return user_approval.lower() == "y"

In [ ]:
system_content="""  You are a helpful assistant with coding capabilities.        Your goal is to help the user accurately and safely with programming tasks  and information retrieval.                ## Core Behavior        - If you already have enough information, answer directly.        - If you do not have enough information, use the search tool before answering.        - If the question requires a date, don't hesitate to ask the user for the date. If it asks for today, call the function get_today_date.        - Base answers only on real retrieved information.        - If information is missing from results, clearly say you could not find it.        - Do NOT hallucinate or guess facts.        - Provide clear, concise, structured responses.                ## Coding Assistant Role        You are also a helpful coding assistant.        Your goal is to help the user with programming tasks such as:        - Writing code        - Editing files        - Running shell commands        - Reading project files        - Debugging issues        ---        ## Available Tools        You have access to the following tools:        1. edit_file: Modify files by replacing text or create new files        2. run_command: Execute shell commands        3. list_directory: View the contents of directories        4. read_file_content: Read the content of files        5. search_internet: Search the internet for relevant and up-to-date information related to the user's query.        6. get_today_date: Return today's date in ISO format (YYYY-MM-DD).        ---        ## Tool Usage Rules        When handling coding tasks:        1. Understand the user request clearly        2. Break complex tasks into smaller steps        3. Use tools to inspect the codebase when needed        4. Implement changes carefully while preserving existing structure and style        5. Explain what you are doing and why        ---        ## Safety & Correctness Rules        - Do not assume or fabricate code behavior        - Always verify changes using tools when possible        - If unsure, ask clarifying questions before making changes        - Be careful with file modifications and shell commands        ---        ## Conversation Rule        After every final response (even after tool call), ask exactly one short, relevant follow-up question to keep the conversation going.                Examples:        - "Would you like me to explain how this works?"        - "Do you want to see a complete example?"        - "Would you like help testing this code?"                ## Execution Requirement        You must run and test your changes using available tools before reporting success.        Do not claim a solution is correct unless it has been verified."""

In [ ]:
import jsondef parse_tool_call(s):    try:        return json.loads(s)    except json.JSONDecodeError:        # Escape literal newlines        s = s.replace("\r", "\\r").replace("\n", "\\n").replace("\t", "\\t")        return json.loads(s)

In [ ]:
import reimport jsondef extract_tool_call(text):    text = tokenizer.decode(text[0], skip_special_tokens=True)    decoded_text = text[text.find("</think>"):]    tool_idx=decoded_text.find("<tool_call>")    if tool_idx==-1:      return False    decoded_text = decoded_text[tool_idx:]    decoded_text = decoded_text.replace("<tool_call>","")    decoded_text = decoded_text.replace("</tool_call>","")    decoded_text = decoded_text.strip()    data = parse_tool_call(decoded_text)    return data

In [ ]:
agent_state = {    "messages": [        {            "role": "system",            "content": system_content,        }    ],}def loop():    counter=1    while True:        if counter==1:            user_input=input("How can I help you?\n")        else:            user_input=input("> ")                    agent_state["messages"].append({"role": "user","content": user_input})               # print(f"[Thinking step {len(agent_state['messages'])}]")        print()        # ✅ build prompt from STATE (not undefined "messages")        text = tokenizer.apply_chat_template(            agent_state["messages"],            tools=tools,            tokenize=False,            add_generation_prompt=True,            enable_thinking=False        )        inputs = tokenizer(text, return_tensors="pt").to(model.device)        decoded = model.generate(**inputs,max_new_tokens=32768,streamer=streamer)                raw_txt = tokenizer.decode(decoded[0], skip_special_tokens=True)        raw_txt = raw_txt[raw_txt.find("</think>"):].replace("</think>","")        # output = model.generate(**inputs,max_new_tokens=32768)        # decoded = tokenizer.decode(output[0], skip_special_tokens=True)        # print("\nMODEL OUTPUT:\n", decoded)                tool_call = extract_tool_call(decoded)        # 1. No tool call → final answer        if not tool_call:            agent_state["messages"].append({                "role": "assistant",                "content": raw_txt.strip()            })        else:            # 2. Tool call exists            name = tool_call["name"]            args = tool_call["arguments"]                print(f"\nTool called: {name}")            print(args)                # =========================            # TOOL EXECUTION            # =========================                if name == "run_command":                if not ask_user_approval("Run command?"):                    result = "Cancelled by user"                else:                    result = run_command(**args)                elif name == "list_directory":                result = list_directory(**args)                        elif name == "search_internet":                result = search_internet(**args)                elif name == "read_file_content":                result = read_file_content(**args)                elif name == "edit_file":                if ask_user_approval("Edit file?"):                    result = edit_file(**args)                else:                    result = "Cancelled"            elif name == "get_today_date":                result = get_today_date()            else:                result = "Unknown tool"                # 3. append tool result            agent_state["messages"].append({                "role": "assistant",                "content":raw_txt.strip()            })                agent_state["messages"].append({                "role": "tool",                "content": str(result),                "name": name            })            print()            agent_state["messages"].append({"role": "user","content": "Okay, what did you do?"})            text = tokenizer.apply_chat_template(               agent_state["messages"],                tools=tools,                tokenize=False,                add_generation_prompt=True,                enable_thinking=False            )            inputs = tokenizer(text, return_tensors="pt").to(model.device)            decoded = model.generate(**inputs,max_new_tokens=32768,streamer=streamer)            raw_txt = tokenizer.decode(decoded[0], skip_special_tokens=True)            raw_txt = raw_txt[raw_txt.find("</think>"):].replace("</think>","")            agent_state["messages"].append({                "role": "assistant",                "content":raw_txt.strip()})        print()        counter+=1        

In [ ]:
# create a python script To calculate the factorial of number passed in stdinloop()

In [ ]:
os.remove("/kaggle/working/factorial.py")

In [ ]:
!pip install streamlit!pip install tavily-python!pip install subprocess.run

In [ ]:
%%writefile app.pyimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport osimport shutilimport richimport requestsimport subprocessfrom datetime import dateimport osfrom transformers import AutoModelForCausalLM, AutoTokenizer,TextStreamerfrom tavily import TavilyClientimport reimport jsonimport streamlit as st@st.cache_resourcedef load_model():    model_name = "Qwen/Qwen3-4B"    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModelForCausalLM.from_pretrained(        model_name,        torch_dtype="auto",        device_map="auto"    )    return tokenizer, modeltokenizer, model = load_model()from threading import Threadfrom transformers import TextIteratorStreamerstreamer = TextIteratorStreamer(    tokenizer,    skip_prompt=True,    skip_special_tokens=True,)# Configure TAVILY_API_KEY as an environment variable or Colab Secret.tavily_api_key = os.getenv("TAVILY_API_KEY")client = TavilyClient(api_key=tavily_api_key) if tavily_api_key else Nonedef edit_file(filename: str, find_str: str, replace_str: str) -> bool:    # If the file doesn't exist and find_str is empty, create the file    if not os.path.exists(filename) and find_str == "":        with open(filename, "w") as f:            f.write(replace_str)        return True    # If the file exists, read it and replace the string    try:        with open(filename, "r") as f:            content = f.read()        # Replace the string        if find_str in content:            new_content = content.replace(find_str, replace_str)            # Write the modified content back to the file            with open(filename, "w") as f:                f.write(new_content)            return True    except FileNotFoundError:        print(f"File {filename} not found and find_str is not empty.")    return Falsedef run_command(command, working_dir) -> tuple[str, int]:    try:        # Set up the process with the working directory if provided        process = subprocess.Popen(            command,            shell=True,            stdout=subprocess.PIPE,            stderr=subprocess.STDOUT,            text=True,            cwd=working_dir,        )        # Get the output and return code        output, _ = process.communicate()        error_code = process.returncode        # Clip output to first 1000 and last 1000 characters if it's longer than 2000 characters        if len(output) > 2000:            output = output[:1000] + "\n\n[...content clipped...]\n\n" + output[-1000:]        return output, error_code    except Exception as e:        return str(e), 1def list_directory(path=".") -> str:    try:        # Get the list of files and directories        items = os.listdir(path)        # Format the output        if not items:            return f"Directory '{path}' is empty."        result = f"Contents of directory '{path}':\n"        for item in items:            full_path = os.path.join(path, item)            if os.path.isdir(full_path):                item_type = "Directory"            else:                item_type = "File"            result += f"- {item} ({item_type})\n"        return result.strip()    except FileNotFoundError:        return f"Error: Directory '{path}' not found."    except PermissionError:        return f"Error: Permission denied to access '{path}'."    except Exception as e:        return f"Error listing directory '{path}': {str(e)}"def read_file_content(path: str) -> str:    try:        with open(path, "r", encoding="utf-8") as file:            content = file.read()            if len(content) > 2000:                # Get first 1000 and last 1000 characters                first_part = content[:1000]                last_part = content[-1000:]                content = first_part + "\n\n[...content clipped...]\n\n" + last_part        return content    except FileNotFoundError:        return f"Error: File '{path}' not found."    except PermissionError:        return f"Error: Permission denied to access '{path}'."    except UnicodeDecodeError:        return f"Error: Unable to decode '{path}'. The file might be binary or use an unsupported encoding."    except Exception as e:        return f"Error reading file '{path}': {str(e)}"def search_internet(query: str) -> str:    """    Retrieve the most relevant text context for the user's query    from the knowledge base.    Args:        user_prompt: The user's search query.    """    if client is None:        return "Internet search is unavailable. Set the TAVILY_API_KEY environment variable."    results = client.search(query=query, max_results=3)    retrieved_context = ""    for c,i in enumerate(results["results"]):        temp_txt=f"source {c+1} : \n"+i["content"]        retrieved_context+=temp_txt+"\n"    return retrieved_contextdef get_today_date() -> str:    """    Return today's date in ISO format (YYYY-MM-DD).    Returns:        str: Today's date.    """    return date.today().isoformat()tools = [    {        "type": "function",        "function": {            "name": "edit_file",            "description": "Apply a diff to a file by replacing occurrences of find_str with replace_str.",            "parameters": {                "type": "object",                "properties": {                    "filename": {                        "type": "string",                        "description": "The name of the file to modify",                    },                    "find_str": {                        "type": "string",                        "description": "The string to find in the file",                    },                    "replace_str": {                        "type": "string",                        "description": "The string to replace with",                    },                },                "required": ["filename", "find_str", "replace_str"],            },        },    },    {        "type": "function",        "function": {            "name": "run_command",            "description": "Run a shell command and return its output and error code.",            "parameters": {                "type": "object",                "properties": {                    "command": {                        "type": "string",                        "description": "The command to run in the shell",                    },                    "working_dir": {                        "type": "string",                        "description": "The working directory to run the command in",                    },                },                "required": ["command", "working_dir"],            },        },    },    {        "type": "function",        "function": {            "name": "list_directory",            "description": "List the contents of a directory.",            "parameters": {                "type": "object",                "properties": {                    "path": {                        "type": "string",                        "description": "The path to the directory to list. Defaults to the current directory.",                    },                },                "required": ["path"],            },        },    },    {        "type": "function",        "function": {            "name": "read_file_content",            "description": "Read and return the content of a file.",            "parameters": {                "type": "object",                "properties": {                    "path": {                        "type": "string",                        "description": "The path to the file to read. Defaults to the current directory.",                    },                },                "required": ["path"],            },        },    },        {        "type": "function",        "function": {            "name": "search_internet",            "description": (                    "Search the internet for up-to-date information and return the most "                    "relevant results. Use this tool whenever the user's question requires "                    "current events, recent news, live information, facts that may have "                    "changed over time, or information that is not available in the conversation."            ),            "parameters": {                "type": "object",                "properties": {                    "query": {                        "type": "string",                        "description": (                            "The search query describing what information to look for on the internet."                        ),                    }                },                "required": ["query"],            },        },    },    {    "type": "function",    "function": {        "name": "get_today_date",        "description": (            "Get today's date in ISO format (YYYY-MM-DD). "            "Use this when the user asks about the current date, today, or time-related queries."        ),        "parameters": {            "type": "object",            "properties": {},            "required": []        },    },}   ]# exit if last message doesn't have any tool calls (agent doesn't want to take any more actions)def is_goal_achieved(state) -> bool:    if len(state["messages"]) <= 2:        return False    last_message = state["messages"][-1]    return (        "role" in last_message        and last_message["role"] == "assistant"        and "tool_calls" not in last_message    )def ask_user_approval(message: str) -> bool:    user_approval = input(f"{message} (y/n): ")    return user_approval.lower() == "y"def extract_tool_call(text):    # text = tokenizer.decode(text[0], skip_special_tokens=True)    # decoded_text = text[text.find("</think>"):]    decoded_text = text    tool_idx=decoded_text.find("<tool_call>")    if tool_idx==-1:      return False    decoded_text = decoded_text[tool_idx:]    decoded_text = decoded_text.replace("<tool_call>","")    decoded_text = decoded_text.replace("</tool_call>","")    decoded_text = decoded_text.strip()    data = parse_tool_call(decoded_text)    return datadef parse_tool_call(s):    try:        return json.loads(s)    except json.JSONDecodeError:        # Escape literal newlines        s = s.replace("\r", "\\r").replace("\n", "\\n").replace("\t", "\\t")        return json.loads(s)system_content="""  You are a helpful assistant with coding capabilities.        Your goal is to help the user accurately and safely with programming tasks  and information retrieval.                ## Core Behavior        - If you already have enough information, answer directly.        - If you do not have enough information, use the search tool before answering.        - If the question requires a date, don't hesitate to ask the user for the date. If it asks for today, call the function get_today_date.        - Base answers only on real retrieved information.        - If information is missing from results, clearly say you could not find it.        - Do NOT hallucinate or guess facts.        - Provide clear, concise, structured responses.        ## Response Formatting        Make responses visually engaging and easy to read.                - Use Markdown formatting extensively.        - Use headings (##, ###) to organize content.        - Use bullet lists and numbered lists where appropriate.        - Use tables whenever they improve readability or compare information.        - Use emojis/icons to highlight sections, such as:          - ✅ Success          - ❌ Errors          - ⚠️ Warnings          - 💡 Tips          - 🔍 Analysis          - 📌 Important          - 🚀 Next Steps          - 🛠️ Commands          - 📄 Files        - Use code blocks with the correct language for all code.        - Highlight important values using **bold** or `inline code`.        - Separate long sections with horizontal rules (`---`).        - Prefer structured layouts over long paragraphs.        - When explaining multiple options, present them in a comparison table when appropriate.        - Summarize long answers with a short "Key Takeaways" section.        - Keep responses professional, clean, and visually appealing without adding unnecessary decorations.        ## Coding Assistant Role        You are also a helpful coding assistant.        Your goal is to help the user with programming tasks such as:        - Writing code        - Editing files        - Running shell commands        - Reading project files        - Debugging issues        ---        ## Available Tools        You have access to the following tools:        1. edit_file: Modify files by replacing text or create new files        2. run_command: Execute shell commands        3. list_directory: View the contents of directories        4. read_file_content: Read the content of files        5. search_internet: Search the internet for relevant and up-to-date information related to the user's query.        6. get_today_date: Return today's date in ISO format (YYYY-MM-DD).        ---        ## Tool Usage Rules        When handling coding tasks:        1. Understand the user request clearly        2. Break complex tasks into smaller steps        3. Use tools to inspect the codebase when needed        4. Implement changes carefully while preserving existing structure and style        5. Explain what you are doing and why        6. After receiving a tool result, evaluate whether the user's request has been fully completed.        7. If additional information or actions are still required, call another tool.        8. Repeat this process until all necessary steps have been completed.        9. Only provide a final answer when no more tool calls are needed.        10. When the task is complete, respond to the user in natural language without wrapping the response in a tool call.        ---        ## Safety & Correctness Rules        - Do not assume or fabricate code behavior        - Always verify changes using tools when possible        - If unsure, ask clarifying questions before making changes        - Be careful with file modifications and shell commands        ---        ## Conversation Rule        After every final response (even after tool call), ask exactly one short, relevant follow-up question to keep the conversation going.                Examples:        - "Would you like me to explain how this works?"        - "Do you want to see a complete example?"        - "Would you like help testing this code?"                ## Execution Requirement        You must run and test your changes using available tools before reporting success.        Do not claim a solution is correct unless it has been verified."""st.set_page_config(page_title="My Chatbot", page_icon="🤖")st.title("🤖 My Chatbot")# Store chat historyif "messages" not in st.session_state:    st.session_state.messages = [{"role": "system","content": system_content}]# Display previous messagesfor message in st.session_state.messages:    if message["content"]=="Is there anything else you need to do? If yes, do it. If no, give me a summary of what you've already done.":        pass    elif message["role"]=="system":        pass    elif message["role"]=="tool":        pass    else:        with st.chat_message(message["role"]):             st.markdown(message["content"])# User inputprompt = st.chat_input("Ask me anything...")if prompt:    # Show user message    st.session_state.messages.append({"role": "user","content": prompt})    with st.chat_message("user"):        st.markdown(prompt)    # Show assistant message    with st.chat_message("assistant"):        messages= st.session_state.messages        text = tokenizer.apply_chat_template(            messages,            tools=tools,            tokenize=False,            add_generation_prompt=True,            enable_thinking=False        )        inputs = tokenizer(text, return_tensors="pt").to(model.device)                generation_kwargs = dict(            **inputs,            streamer=streamer,            max_new_tokens=32768,        )                thread = Thread(target=model.generate, kwargs=generation_kwargs)        thread.start()                raw_txt = st.write_stream(streamer)        for step in range(5):            tool_call = extract_tool_call(raw_txt)            if not tool_call:                 st.session_state.messages.append({"role": "assistant","content": raw_txt.strip()})                 break            else:                # 2. Tool call exists                name = tool_call["name"]                args = tool_call["arguments"]                                if name == "run_command":                    result = run_command(**args)                        elif name == "list_directory":                    result = list_directory(**args)                                elif name == "search_internet":                    result = search_internet(**args)                        elif name == "read_file_content":                    result = read_file_content(**args)                                elif name == "edit_file":                    result = edit_file(**args)                                elif name == "get_today_date":                    result = get_today_date()                                else:                    result = "Unknown tool"                            # 3. append tool result                st.session_state.messages.append({"role": "assistant","content":raw_txt.strip()})                        st.session_state.messages.append({"role": "tool","content": str(result),"name": name})                        st.session_state.messages.append({"role": "user","content": "Is there anything else you need to do? If yes, do it. If no, give me a summary of what you've already done."})                                messages= st.session_state.messages                text = tokenizer.apply_chat_template(                    messages,                    tools=tools,                    tokenize=False,                    add_generation_prompt=True,                    enable_thinking=False                )                                inputs = tokenizer(text, return_tensors="pt").to(model.device)                generation_kwargs = dict(**inputs,streamer=streamer,max_new_tokens=32768)                            thread = Thread(target=model.generate, kwargs=generation_kwargs)                thread.start()                            raw_txt = st.write_stream(streamer)                        st.session_state.messages.append({"role": "assistant","content":raw_txt.strip()})

In [ ]:
import subprocessimport timestreamlit = subprocess.Popen(    [        "streamlit",        "run",        "app.py",        "--server.port",        "8501",        "--server.headless",        "true",    ])time.sleep(5)import subprocessimport repinggy = subprocess.Popen(    [        "ssh",        "-o", "StrictHostKeyChecking=no",        "-p", "443",        "-R", "0:localhost:8501",        "a.pinggy.io"    ],    stdout=subprocess.PIPE,    stderr=subprocess.STDOUT,    text=True)public_url = Nonefor line in iter(pinggy.stdout.readline, ""):    print(line.strip())    match = re.search(r"https://[a-zA-Z0-9.-]+\.pinggy\.link", line)    if match:        public_url = match.group(0)        breakprint("Public URL:", public_url)